# Quality of Life

> Functions to improve quality of life. Duplicates a subset of those in `EnvDL`'s `core`.

In [ ]:
#| default_exp qol

In [ ]:
#| export
import pandas as pd
import pyarrow.parquet as pq

In [ ]:
#| hide
from nbdev.showdoc import *

## Functions for working with files, directories with the intention of caching results.

In [ ]:
#| export
def read_txt(path, 
             **kwargs # Intended to allow for explicit 'encoding' to be passed into open the file
            ):
    if 'encoding' in kwargs.keys():
        print(kwargs)
        with open(path, 'r', encoding  = kwargs['encoding']) as f:
            data = f.read()        
    else:    
        with open(path, 'r') as f:
            data = f.read()
            
    return(data)

In [ ]:
#| export

def print_txt(path):
    print(read_txt(path = path))

In [ ]:
#| export

def read_json(json_path
             ):
    "Read and return json. Used for train/validation/test splits"
    import json
    with open(json_path, 'r') as fp:
        dat = json.load(fp)
    return(dat)

In [ ]:
#| export

def write_json(obj, json_path):
    import json
    with open(json_path, 'w') as f:
        f.write(json.dumps(obj, indent=4, sort_keys=True))

In [ ]:
#| export

def read_pq_or_pd(file_path:str):
    "Using file extention read a parquet -> csv -> table"
    match file_path.split('.')[-1]:
        case 'parquet': 
            out = pq.read_table(file_path).to_pandas()
        case 'csv': 
            out = pd.read_csv(file_path)
        case _: 
            out = pd.read_table(file_path)
    return out


In [ ]:
#| export

def ensure_dir_path_exists(dir_path = '../ext_data' # Directory path to check
                          ):
    "Iteratively check for and create directories to store output. Ideally this would just be os.mkdirs() but that function is not available in this version of python"
    import os
    
    for i in range(2, len(dir_path.split('/'))+1):
        path_part = '/'.join(dir_path.split('/')[0:i])
        if not os.path.exists(path_part):
            os.mkdir(path_part)

In [ ]:
#| export

"Retrieve a previously calculated result. Return None if it cannot be found."
def get_cached_result(
    save_path
):
    import os
    import pickle as pkl
#     import pickle5 as pkl # Using non-base version of pickle 
#                           # conda env with gpu support for tf and torch uses python 3.7.
#                           # Python 3.7 doesn't contain pickle v 5
    if not os.path.exists(save_path):
        cached_result = None
    else:
        with open(save_path, 'rb') as handle:
                cached_result = pkl.load(handle)
    return(cached_result)

In [ ]:
#| export

def put_cached_result(
    save_path,
    save_obj
):
    import pickle as pkl
#     import pickle5 as pkl
#     from EnvDL.core import ensure_dir_path_exists
    ensure_dir_path_exists(dir_path= '/'.join(save_path.split('/')[:-1]) )
    
    with open(save_path, 'wb') as handle:
            pkl.dump(save_obj, 
                     handle, 
                     protocol=4 # version 4 is used instead of 5 because the container
                                # I'm using with tf and torch uses python 3.7 and version
                                # 5 is introduced in 3.8
                    )

In [ ]:
#| export

def remove_matching_files(
    cache_path, # Directory to query
    match_regex_list = ['.*\.pt', 'yhats\.csv', 'loss_df\.csv'], # List of regexes to match (okay if two regexes match the same entry)
    dry_run = True # Print files to be deleted or delete them. 
):
    "Helper function to clear out cache. Remove files from a folder if they match one of a given set of regexes. Ignores directories in directory. Useful for clearing out model artifacts."
    import os
    import re
    # if empty set is provided, match nothing.
    if match_regex_list == []:
        match_regex_list = ['']
    
    files_to_remove = [[e for e in os.listdir(cache_path) if re.match(match_regex, e)
                       ] for match_regex in match_regex_list]
    # make a (potential) list of lists into a flat list
    new_list = []
    for sub_list in files_to_remove:
        new_list = new_list + sub_list
    # ensure it's deduplicated in case two regexes match with the same item
    files_to_remove = list(set(new_list))
    # remove any directories from consideration
    files_to_remove = [e for e in files_to_remove if os.path.isfile(cache_path+e)]
    # sort to make output more pleasant
    files_to_remove.sort()

    if files_to_remove == []:
        print('No files found to remove.')
    else:
        if dry_run:
            print('Command would remove:')
            print('\n'.join(files_to_remove))
        else:
            for file in files_to_remove:
                os.remove(cache_path+file)

# remove_matching_files(
#     cache_path,
#     match_regex_list = ['.*\.pt', 'yhats\.csv', 'loss_df\.csv'],
#     dry_run = False
# )

### Default parameters for tuning and training

In [ ]:
#| export
def params_data(): return {
    'species': None,
    'num_nucleotides': 4,

    ## Cache intermediate
    # 'use_data_cache': True, 

    ## paths
    'graph_cache_path': './', # kegg_cache_path
    'gff_path': None,
    'hmp_path': None,
    'phno_path': None,
    'cache_path': './',
    'model_path': None,

    ## Graph
    'graph_source': 'kegg', # kegg, cxn
    'kegg_catalog': '00001',
    'graph_cxn'   :  None, 
    ##
    'holdout_type': 'percent', # percent, taxa, parent
    'holdout_percent': 0.2,
    'holdout_seed': 8923747,
    'holdout_taxa': [],
    'holdout_taxa_ignore': [], # taxa to drop. Useful for keeping a test set truely separate
    ## 
    'dataloader_shuffle_train': True,
    'dataloader_shuffle_valid': False,
}

In [ ]:
#| export
def params_run(): return {
    'use_data_cache': True, # save and restore from vnn cache?
    'patch'     : False, # Should code be patched using the code in vnn_patch.py?
    
    'batch_size': 32,
    'max_epoch' : 2,
    'run_mode'  : 'setup', # modes: tune, train, predict, eval
    ## tune ====
    'tune_trials': 1,
    'tune_max'  : 1,
    'tune_force': False, # should we run more trials even if the target number has been reached?

    ## train ====
    'train_from_ax' : False, # should we use the best ax trial or a user specified network?
    'train_save': True,
    'train_name': '',
    ## predict ====
    ## eval ====
    'eval': [],
}

In [ ]:
#| export
def params(): return {
    'default_out_nodes_inp'  : 1,
    'default_out_nodes_edge' : 1,
    'default_out_nodes_out'  : 1, #NOTE This will be overwritten below to match y.shape[1]

    'default_drop_nodes_inp' : 0.0,
    'default_drop_nodes_edge': 0.0,
    'default_drop_nodes_out' : 0.0,

    'default_reps_nodes_inp' : 1,
    'default_reps_nodes_edge': 1,
    'default_reps_nodes_out' : 1,

    'default_decay_rate'     : 0
    }

In [ ]:
#| export
def params_list(): return [    
    ## Output Size ====
    {
    'name': 'default_out_nodes_inp',
    'type': 'range',
    'bounds': [1, 8],
    'value_type': 'int',
    'log_scale': False
    },
    {
    'name': 'default_out_nodes_edge',
    'type': 'range',
    'bounds': [1, 32],
    'value_type': 'int',
    'log_scale': False
    },
    {
    'name': 'default_out_nodes_out',
    'type': 'fixed',
    'value': 1, #NOTE This will be overwritten below to match y.shape[1]
    'value_type': 'int',
    'log_scale': False
    },
    ## Dropout ====
    {
    'name': 'default_drop_nodes_inp',
    'type': 'range',
    'bounds': [0.01, 0.99],
    'value_type': 'float',
    'log_scale': False
    },
    {
    'name': 'default_drop_nodes_edge',
    'type': 'range',
    'bounds': [0.01, 0.99],
    'value_type': 'float',
    'log_scale': False
    },
    {
    'name': 'default_drop_nodes_out',
    'type': 'range',
    'bounds': [0.01, 0.99],
    'value_type': 'float',
    'log_scale': False,
    'sort_values':True
    },
    ## Node Repeats ====
    {
    'name': 'default_reps_nodes_inp',
    'type': 'choice',
    'values': [1, 2, 3],
    'value_type': 'int',
    'is_ordered': True,
    'sort_values':True
    },
    {
    'name': 'default_reps_nodes_edge',
    'type': 'choice',
    'values': [1, 2, 3],
    'value_type': 'int',
    'is_ordered': True,
    'sort_values':True
    },
    {
    'name': 'default_reps_nodes_out',
    'type': 'choice',
    'values': [1, 2, 3],
    'value_type': 'int',
    'is_ordered': True,
    'sort_values':True
    },
    ## Node Output Size Scaling ====
    {
    'name': 'default_decay_rate',
    'type': 'choice',
    'values': [0+(0.1*i) for i in range(10)]+[1.+(1*i) for i in range(11)],
    'value_type': 'float',
    'is_ordered': True,
    'sort_values':True
    }
    ]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()